# Team T69 ? extended TRIDENT v1 training
New experiment initialized from verified B0, not a rerun of B0/A1.
**Budget: 12 epochs, batch 1, Adam 1e-4, FP32, seed 69.** Each epoch has 672 training updates (96 per task) and 861 validation inferences (123 original frames x seven tasks).
Dense SR at 2x/4x/8x is separate from target-grid sparse/noisy/hole completion.

Select **Runtime ? Change runtime type ? T4 GPU**. Upload the supplied current source ZIP, development-only data ZIP and B0 checkpoint into the Drive folder below, then **Run all**.
Training is authorized by the current user request, but cannot begin until the full test suite and full-resolution, zero-update GPU preflight pass for this exact code/config/data/environment.
No official test payload is included, loaded or evaluated. There is no export functionality. Nothing automatically extends the 12-epoch budget.

## 1. Paths and approved budget
For recovery, set RESUME_CHECKPOINT to the last complete extended-model epoch file from this same run. Keep the same inputs/configuration. Leave None for a new run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os,sys,json,hashlib,zipfile,subprocess,uuid,datetime
from pathlib import Path,PurePosixPath
os.environ['PYTHONDONTWRITEBYTECODE']='1'
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
INPUT=Path('/content/drive/MyDrive/TRIDENT-T69/extended_v1_inputs')
SOURCE_ZIP=INPUT/'trident_extended_v1_current_source.zip'
DATA_ZIP=INPUT/'nyu_development795_only.zip'
B0_CHECKPOINT=INPUT/'B0_epoch_002.pt'
RESUME_CHECKPOINT=None  # e.g. Path('/content/drive/MyDrive/TRIDENT-T69/runs/extended_v1/<run>/epoch_003.pt')
TRAINING_APPROVED=True  # current request explicitly authorizes this new, verified 12-epoch run
RUN_ID=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+uuid.uuid4().hex[:10]
WORK=Path('/content')/('trident_extended_'+RUN_ID)
REPO,DATA=WORK/'repo',WORK/'development_data'
RUNS=Path('/content/drive/MyDrive/TRIDENT-T69/runs/extended_v1')
OUTPUT=Path(RESUME_CHECKPOINT).parent if RESUME_CHECKPOINT else RUNS/RUN_ID
EVIDENCE=Path('/content/drive/MyDrive/TRIDENT-T69/preflight/extended_v1')/RUN_ID
PY=sys.executable
print('Output:',OUTPUT)

## 2. Require Tesla T4 and verify immutable input SHA256

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a CUDA GPU runtime'
assert torch.cuda.device_count()==1 and torch.cuda.get_device_name(0)=='Tesla T4', 'This configuration requires one Tesla T4'
def digest(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()
EXPECTED={SOURCE_ZIP:'b2836c7dc202acf2050bba4780806e571e7995170387888e37fc6e11a105d232',DATA_ZIP:'726863d8df28f453c77feeabcbe94f77aa39be9ba52e863a6da81348c9a5cf92',
          B0_CHECKPOINT:'a6416c47a16ec78e97c8b734ae7fb23cba4dbbeb5f5564a3e02d61a4a159c1b2'}
for p,h in EXPECTED.items():
    assert p.is_file(),f'Missing input: {p}'
    assert digest(p)==h,f'SHA256 mismatch: {p}'
    print(p.name,h)
assert not WORK.exists() and not EVIDENCE.exists()
if RESUME_CHECKPOINT:
    assert Path(RESUME_CHECKPOINT).is_file() and 'extended_v1' in OUTPUT.parts
else:
    assert not OUTPUT.exists(), 'Never reuse an existing new-run directory'
print('Torch:',torch.__version__,'CUDA:',torch.version.cuda,'GPU:',torch.cuda.get_device_name(0))

## 3. Extract only current source and development payloads
The data ZIP contains 795 development frames; locked test samples are absent. Mixed manifest labels are retained unchanged to verify the original split.

In [ ]:
WORK.mkdir();REPO.mkdir();DATA.mkdir();EVIDENCE.mkdir(parents=True)
def safe_names(z):
    names=z.namelist()
    assert len(names)==len(set(names))
    for item in z.infolist():
        p=PurePosixPath(item.filename)
        assert not p.is_absolute() and '..' not in p.parts and '\\' not in item.filename
        assert ((item.external_attr>>16)&0o170000)!=0o120000,'Symlinks forbidden'
    return set(names)
with zipfile.ZipFile(SOURCE_ZIP) as z:
    safe_names(z);z.extractall(REPO)
with zipfile.ZipFile(DATA_ZIP) as z:
    names=safe_names(z)
    manifest=z.read('manifest_development.json')
    assert hashlib.sha256(manifest).hexdigest()=='d1cdbcf0ff3ee96f2e0c173c3c4701ed3f5c526ab517ce3b988268c3c6a391f5'
    all_rows=json.loads(manifest)
    rows=[r for r in all_rows if r['split'] in ('train','validation')]
    assert sum(r['split']=='train' for r in rows)==672
    assert sum(r['split']=='validation' for r in rows)==123
    allowed={'manifest_development.json'}|{r[k] for r in rows for k in ('rgb','depth','mask')}
    assert names==allowed and len(names)==2386,'Archive contains non-development payloads'
    z.extractall(DATA)
CONFIG=REPO/'configs/extended_v1/train.json'
assert digest(CONFIG)=='3158a0432076a833f876682aa55777c71191bb8bf2680e91ce576ee73b9590ea'
os.chdir(REPO)
print(CONFIG.read_text())

## 4. Preserve CUDA PyTorch; install additions and run the full test suite
No virtual environment, ensurepip, CPU torch replacement or training command is used during setup. The optional original-ZIP-based B0 parity test can skip; actual raw B0 loading is checked in the real-sample preflight.

In [ ]:
# Use Colab's existing runtime directly: no venv or ensurepip.
import tempfile
PY = sys.executable
print("Colab Python:", PY)
torch_probe = """
import json, torch
print(json.dumps(dict(version=str(torch.__version__), cuda=torch.version.cuda,
 location=torch.__file__, cuda_available=torch.cuda.is_available())))
"""
TORCH_BEFORE = json.loads(subprocess.check_output([str(PY), "-c", torch_probe], text=True))
assert TORCH_BEFORE["cuda_available"], "Select a GPU runtime; Colab CUDA PyTorch must already be available"

# Resolve only the additional requirements first. Reject any plan that would
# touch PyTorch or its CUDA stack, then install exactly that plan without resolving
# further dependencies. This cannot pull in a CPU-only torch wheel.
with tempfile.TemporaryDirectory(prefix="trident-pip-") as temporary:
    report = Path(temporary) / "install-plan.json"
    subprocess.run([str(PY), "-m", "pip", "install", "--dry-run",
                    "--only-binary=:all:", "--report", str(report),
                    "-r", str(REPO / "requirements-colab.txt")], check=True)
    plan = json.loads(report.read_text())["install"]
    protected = {"torch", "torchvision", "torchaudio", "torchtext", "triton", "pytorch-triton"}
    for item in plan:
        name = item["metadata"]["name"].lower().replace("_", "-")
        assert name not in protected and not name.startswith("nvidia-"), f"Refusing to replace Colab GPU package: {name}"
    urls = [item["download_info"]["url"] for item in plan]
    if urls:
        subprocess.run([str(PY), "-m", "pip", "install", "--no-deps",
                        "--only-binary=:all:", *urls], check=True)
subprocess.run([str(PY), "-m", "pip", "install", "--no-deps", "--no-build-isolation", "-e", str(REPO)], check=True)
TORCH_AFTER = json.loads(subprocess.check_output([str(PY), "-c", torch_probe], text=True))
assert TORCH_AFTER == TORCH_BEFORE, "Colab PyTorch/CUDA changed during setup"
probe = """
import json, platform, torch, numpy, scipy, cv2, PIL, h5py
assert (2, 6) <= tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2]) < (3, 0)
print(json.dumps(dict(python=platform.python_version(), torch=str(torch.__version__),
 cuda=torch.version.cuda, numpy=numpy.__version__, scipy=scipy.__version__,
 opencv=cv2.__version__, pillow=PIL.__version__, h5py=h5py.__version__,
 gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
 gpu_memory_bytes=torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else None)))
"""
ENVIRONMENT = json.loads(subprocess.check_output([str(PY), "-c", probe], text=True))
print(json.dumps(ENVIRONMENT, indent=2))
assert ENVIRONMENT["gpu"], "No CUDA GPU available: select a GPU runtime. Do not run the pilot on CPU."
# Complete project suite after installation; CPU tests do not train a model.
subprocess.run([str(PY), "-m", "pytest", "-q"], check=True,
               env=dict(os.environ, CUDA_VISIBLE_DEVICES=""))

## 5. Verify all 795 development payloads, then qualify seven real training-sample backward passes
This command performs **zero optimizer steps**, checks every trainable gradient, verifies unchanged weights, and records full-resolution T4 memory. Validation payloads are hashed against the original reference, but no validation examples are trained on.

In [ ]:
PREFLIGHT=EVIDENCE/'gpu_preflight.json'
command=[PY,'-B','-m','trident_depth.extended_v1.preflight',
         '--config',str(CONFIG),'--manifest',str(DATA/'manifest_development.json'),
         '--checkpoint',str(B0_CHECKPOINT),'--device','cuda','--output',str(PREFLIGHT)]
subprocess.run(command,cwd=REPO,check=True)
qualification=json.loads(PREFLIGHT.read_text())
assert qualification['status']=='PASS' and qualification['optimizer_steps']==0
assert qualification['gpu']=='Tesla T4' and qualification['full_resolution']
assert qualification['split_audit']['official_test_sample_files_opened']==0
peaks=[r['peak_allocated_bytes'] for r in qualification['results']]
peak=max(peaks)
vram=torch.cuda.get_device_properties(0).total_memory
assert peak+256*1024**2 < .90*vram,'Insufficient measured training memory headroom; stop and report, do not alter batch/config silently'
print('Measured full-resolution preflight GPU peak GiB:',peak/1024**3)
mean_seconds=sum(r['seconds'] for r in qualification['results'])/7
print('Rough 12-epoch forward/backward compute projection, hours:',12*672*mean_seconds/3600)
print('Projection excludes data I/O, optimizer overhead, 861 validation inferences per epoch, metrics, checkpoint writes and interruptions. Actual first-epoch wall time is authoritative.')
with (EVIDENCE/'pip_freeze.txt').open('x') as f:
    subprocess.run([PY,'-m','pip','freeze'],stdout=f,check=True)
with (EVIDENCE/'environment.json').open('x') as f:
    json.dump(dict(environment=ENVIRONMENT,inputs={str(p):h for p,h in EXPECTED.items()},
                   config_sha256=digest(CONFIG),output=str(OUTPUT)),f,indent=2)

## 6. Start or resume only the verified extended experiment
This is the first cell that performs dataset training. A new run never writes into any A1, B0, hybrid or previous extended-run directory. Resume restores the same run at a complete epoch boundary, preserving its approved 12-epoch ceiling.

In [ ]:
assert TRAINING_APPROVED,'Training disabled'
assert torch.cuda.is_available() and torch.cuda.get_device_name(0)=='Tesla T4'
for p,h in EXPECTED.items():assert digest(p)==h,f'Input changed: {p}'
command=[PY,'-B','-m','trident_depth.extended_v1.train','--execute-training',
         '--config',str(CONFIG),'--manifest',str(DATA/'manifest_development.json'),
         '--checkpoint',str(B0_CHECKPOINT),'--preflight',str(PREFLIGHT),'--output',str(OUTPUT)]
if RESUME_CHECKPOINT:command+=['--resume',str(RESUME_CHECKPOINT)]
with (EVIDENCE/'launch.json').open('x') as f:json.dump(dict(command=command),f,indent=2)
with (EVIDENCE/'training_console.log').open('x') as log:
    process=subprocess.Popen(command,cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout:
        print(line,end='');log.write(line);log.flush()
    returncode=process.wait()
assert returncode==0,f'Training stopped. Preserve logs and resume from last complete checkpoint; do not change config. See {EVIDENCE}'

## 7. Inspect validation-selected checkpoint metadata and stop
No export or official test evaluation follows training. Share the run and preflight records for review before a new benchmark or further training.

In [ ]:
completed=sorted(OUTPUT.glob('epoch_*_metrics.json'))
assert completed,'No completed epoch metrics found'
latest=json.loads(completed[-1].read_text())['summary']
print(json.dumps(latest,indent=2))
best=OUTPUT/('epoch_%03d.pt'%latest['best_epoch'])
assert best.is_file()
print('Validation-selected checkpoint:',best)
print('Checkpoint SHA256:',digest(best))
print('Run:',OUTPUT,'Preflight and console evidence:',EVIDENCE)
print('Approved training complete. No automatic extra epochs, inference export or official test evaluation.')
for p,h in EXPECTED.items():assert digest(p)==h,f'Original input changed: {p}'
assert latest['split_audit']['official_test_sample_files_opened']==0
assert not latest['split_audit']['forbidden_access_attempts']
